# Fuzzy AHP Priority Ranking - Sample Development Pipeline

## Decision Support Stage

This notebook documents the Fuzzy AHP calculation pipeline for SentiRank. Phase 10C uses sample development judgements only to validate template conversion, Fuzzy AHP calculation, output export, and figure generation.

**Important:** outputs in this notebook are marked `sample_development_only` and `not_final_expert_judgement`. They are not final Fuzzy AHP weights and must not be used as thesis ranking results.

## Fuzzy AHP Role in SentiRank

Fuzzy AHP is compared with standard AHP because expert judgement can be uncertain. It represents linguistic pairwise judgement with triangular fuzzy numbers and then defuzzifies weights using the centroid method.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "ml-service").exists()
            and (candidate / "datasets").exists()
            and (candidate / "docs").exists()
        ):
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
TEMPLATE_DIR = PROJECT_ROOT / "docs" / "templates" / "ahp"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ML_SERVICE_DIR:", ML_SERVICE_DIR)

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print(f"Missing JSON: {path}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path):
    if not path.exists():
        print(f"Missing CSV: {path}")
        return None
    return pd.read_csv(path)


def display_image(path: Path) -> None:
    if not path.exists():
        print(f"Missing figure: {path}")
        return
    display(Image(filename=str(path)))


def run_ml_script(script_name: str) -> None:
    command = [sys.executable, "scripts/" + script_name]
    result = subprocess.run(
        command,
        cwd=ML_SERVICE_DIR,
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Script failed: {script_name}")

## Final Candidate Criteria

The same five criteria used by AHP are used by Fuzzy AHP so that both methods can be compared directly.

In [ ]:
criteria_payload = load_json(TEMPLATE_DIR / "final_criteria_for_ahp.json")
if criteria_payload:
    criteria_df = pd.DataFrame(criteria_payload["criteria"])
    display(criteria_df[["criterion_id", "criterion_name", "description", "source_labels", "expert_validation_required"]])

## Sample Fuzzy AHP Input

The sample input contains exactly 10 fuzzy pairwise comparisons. Numeric TFN columns are the source of truth for the script.

In [ ]:
fuzzy_sample_input = TEMPLATE_DIR / "sample_development" / "fuzzy_ahp_pairwise_sample_development.csv"
fuzzy_input_df = load_csv(fuzzy_sample_input)
if fuzzy_input_df is not None:
    display(fuzzy_input_df)
    print("Pairwise comparisons:", len(fuzzy_input_df))

## Run Sample Fuzzy AHP Calculation

This cell runs only the sample-development script. It writes to `datasets/outputs/eda/07_fuzzy_ahp/sample_development/` and does not create final `fuzzy_ahp_weights.csv` or ranking outputs.

In [ ]:
run_ml_script("calculate_fuzzy_ahp_from_expert_judgement.py")

## Sample Fuzzy AHP Outputs

The outputs below show fuzzy weights, defuzzified/normalized weights, and modal consistency ratio.

In [ ]:
FUZZY_OUTPUT_DIR = PROJECT_ROOT / "datasets" / "outputs" / "eda" / "07_fuzzy_ahp" / "sample_development"
FUZZY_FIGURE_DIR = PROJECT_ROOT / "docs" / "figures" / "07_fuzzy_ahp" / "sample_development"

weights_df = load_csv(FUZZY_OUTPUT_DIR / "fuzzy_ahp_weights_sample_development.csv")
if weights_df is not None:
    display(weights_df.sort_values("rank"))

modal_consistency = load_json(FUZZY_OUTPUT_DIR / "fuzzy_ahp_modal_consistency_sample_development.json")
if modal_consistency:
    display(pd.DataFrame([modal_consistency]))

matrix_payload = load_json(FUZZY_OUTPUT_DIR / "fuzzy_ahp_pairwise_matrix_sample_development.json")
if matrix_payload:
    display(pd.DataFrame(matrix_payload["modal_crisp_matrix"]))

## Sample Fuzzy AHP Weight Figure

In [ ]:
display_image(FUZZY_FIGURE_DIR / "fuzzy_ahp_weights_sample_development.png")

## Interpretation Notes

- The sample modal CR is expected to be approximately `0` because modal values follow the consistent development AHP matrix.
- Real expert judgement is still required before final Fuzzy AHP weighting.
- Final Fuzzy AHP weights and priority rankings are intentionally not generated in Phase 10C.